# ★ BrandMind AI — Week 9
## Feedback Intelligence & Model Refinement
**Tools:** Pandas · NLTK · Plotly · scikit-learn

In [ ]:
!pip install pandas nltk plotly scikit-learn joblib -q

In [ ]:
import pandas as pd, numpy as np, json, os, datetime, joblib
import matplotlib.pyplot as plt
import nltk
nltk.download(['vader_lexicon','punkt','punkt_tab'], quiet=True)
from nltk.sentiment import SentimentIntensityAnalyzer
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
sia = SentimentIntensityAnalyzer()
print('Ready')

## 1. Feedback Data Structure

In [ ]:
FEEDBACK_CSV = 'data/feedback_log.csv'

def save_feedback(company, logo_r, slogan_r, animation_r, campaign_r, overall_r, suggestion=''):
    sentiment = sia.polarity_scores(suggestion) if suggestion else {'compound': 0, 'pos': 0, 'neg': 0, 'neu': 1}
    row = {
        'timestamp':        datetime.datetime.now().isoformat(),
        'company':          company,
        'logo_rating':      logo_r,
        'slogan_rating':    slogan_r,
        'animation_rating': animation_r,
        'campaign_rating':  campaign_r,
        'overall_rating':   overall_r,
        'suggestion':       suggestion,
        'sentiment_compound': sentiment['compound'],
        'sentiment_positive': sentiment['pos'],
        'sentiment_negative': sentiment['neg'],
    }
    df_new = pd.DataFrame([row])
    if os.path.exists(FEEDBACK_CSV):
        df_new.to_csv(FEEDBACK_CSV, mode='a', header=False, index=False)
    else:
        df_new.to_csv(FEEDBACK_CSV, index=False)
    return row

# Simulate feedback from 50 users
np.random.seed(42)
for i in range(50):
    suggestions = ['Great logo options!', 'Slogans need more punch',
                   'Love the colour palettes', 'Campaign predictions very useful', '',
                   'More font choices please', 'Excellent overall tool', '']
    save_feedback(
        company=f'Company_{i}',
        logo_r=np.random.randint(3,6), slogan_r=np.random.randint(2,6),
        animation_r=np.random.randint(3,6), campaign_r=np.random.randint(3,6),
        overall_r=np.random.randint(3,6), suggestion=np.random.choice(suggestions)
    )
print(f'Generated feedback log: {FEEDBACK_CSV}')
df_fb = pd.read_csv(FEEDBACK_CSV)
print(df_fb.shape)
print(df_fb.describe().round(2))

## 2. Feedback Analytics Dashboard

In [ ]:
df_fb = pd.read_csv(FEEDBACK_CSV)
rating_cols = ['logo_rating','slogan_rating','animation_rating','campaign_rating','overall_rating']
avg_ratings = df_fb[rating_cols].mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('BrandMind AI — Feedback Analytics Dashboard', fontsize=14, fontweight='bold')

# Bar chart - average ratings
ax = axes[0,0]
labels = ['Logo','Slogan','Animation','Campaign','Overall']
colors = ['#b8975a' if v >= 4 else '#6b6a65' for v in avg_ratings]
bars = ax.bar(labels, avg_ratings, color=colors)
ax.set_ylim(0, 5.5); ax.axhline(4, color='#2e7d52', ls='--', lw=1, label='Target 4.0')
ax.set_title('Average Ratings per Asset'); ax.legend(fontsize=8)
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, avg_ratings):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.1, f'{val:.2f}', ha='center', fontsize=9)

# Rating distributions
for ax, col, label in zip([axes[0,1], axes[0,2], axes[1,0], axes[1,1]],
                            ['logo_rating','slogan_rating','animation_rating','campaign_rating'],
                            ['Logo','Slogan','Animation','Campaign']):
    df_fb[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='#1a1a18')
    ax.set_title(f'{label} Rating Distribution')
    ax.set_xlabel('Stars'); ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='x', rotation=0)

# Sentiment analysis
ax = axes[1,2]
df_fb['sentiment_category'] = df_fb['sentiment_compound'].apply(
    lambda x: 'Positive' if x > 0.05 else ('Negative' if x < -0.05 else 'Neutral'))
df_fb['sentiment_category'].value_counts().plot(kind='pie', ax=ax,
    colors=['#2e7d52','#c0392b','#b8975a'], autopct='%1.1f%%')
ax.set_title('Feedback Sentiment Analysis')

plt.tight_layout(); plt.savefig('outputs/feedback_dashboard.png', dpi=150); plt.show()
print('Saved: outputs/feedback_dashboard.png')

## 3. Model Refinement Based on Feedback

In [ ]:
avg = df_fb[rating_cols].mean()

# Determine which modules need refinement
refinement_plan = {}
for col, label in zip(rating_cols[:-1], ['Logo CNN','Slogan NLP','Animation','Campaign RF']):
    rating = avg[col]
    if rating < 3.5:
        action = 'CRITICAL — retrain model with augmented data'
    elif rating < 4.0:
        action = 'Adjust generation parameters / temperature'
    else:
        action = 'Performing well — minor fine-tuning only'
    refinement_plan[label] = {'avg_rating': round(rating,3), 'action': action}
    print(f'{label:20s}: {rating:.2f}/5 → {action}')

# Save refinement plan
with open('outputs/model_refinement_plan.json', 'w') as f:
    json.dump(refinement_plan, f, indent=2)
print('\nSaved: outputs/model_refinement_plan.json')

## Week 9 Complete ✅
- ✅ Feedback forms (logo, slogan, animation, campaign — 1-5 scale)
- ✅ NLTK VADER sentiment analysis on suggestions
- ✅ Plotly/Matplotlib feedback analytics dashboard
- ✅ Model refinement plan based on aggregated ratings
- ✅ Feedback stored in CSV for retraining